In [1]:
import skimage.io
import skimage.transform
import os
import shutil
import random
import joblib
import constants

In [2]:
def img_to_512(dir_imgs, dir_labels, new_img_path, new_label_path):
    
    # Remove the directories so we can stat anew
    if os.path.isdir(new_img_path):
        shutil.rmtree(new_img_path)
    os.makedirs(new_img_path)
    if os.path.isdir(new_label_path):
        shutil.rmtree(new_label_path)
    os.makedirs(new_label_path)


    aantal = len(os.listdir(os.path.join(os.getcwd(), dir_imgs)))
    teller_files = 1
    for filename in os.listdir(os.path.join(os.getcwd(), dir_imgs)):
        print(f"{teller_files}/{aantal}", end='\r')
        filename = filename.split(".")[0]
        #Per image
        img = skimage.io.imread(f"{dir_imgs}/{filename}.JPG")
        labels = open(f"{dir_labels}/{filename}.txt")
        lines = labels.readlines()
        lines = [line.rstrip() for line in lines]
        teller = 1
        y = 0
        while(y < constants.Y_END - constants.WIDTH_PIECE): #zijkanten worden weggegooid indien te klein
            x = 0
            while(x < constants.X_END - constants.WIDTH_PIECE): #zijkanten worden weggegooid indien te klein
                x_end = x+constants.WIDTH_PIECE if x+constants.WIDTH_PIECE < constants.X_END else constants.X_END
                y_end = y+constants.WIDTH_PIECE if y+constants.WIDTH_PIECE < constants.Y_END else constants.Y_END
                # print(f"x: {x}\ty: {y}\tx_end: {x_end}\ty_end: {y_end}")
                img_piece = img[y:y_end,x:x_end]
                labels_piece = open(f"{new_label_path}/{filename}_{teller}.txt", "a")
                for line in lines:
                    name, x_center, y_center, width, height = [round(float(s)) for s in line.split("\t")]
                    if x_center >= x and x_center < x_end and y_center >= y and y_center < y_end:
                        labels_piece.write("0\t"+str(x_center-x)+"\t"+str(y_center-y)+"\t"+str(width)+"\t"+str(height)+"\n")
                labels_piece.close()
                if os.stat(labels_piece.name).st_size == 0: os.remove(labels_piece.name)
                # print(img_piece)
                skimage.io.imsave(f"{new_img_path}/{filename}_{teller}.JPG", img_piece)
                x += constants.WIDTH_PIECE - constants.OVERLAP #+550
                teller += 1
            y += constants.WIDTH_PIECE - constants.OVERLAP
        labels.close()
        teller_files+= 1

In [3]:
def split_clustering(path, sorted_path):
    # Remove the directories so we can stat anew
    if os.path.isdir(sorted_path):
        shutil.rmtree(sorted_path)
    os.makedirs(sorted_path)
    os.makedirs(f"{sorted_path}/positive")
    os.makedirs(f"{sorted_path}/negative")

    kmeans = joblib.load('./trained_models/kmeans')

    img_dir = os.listdir(path)
    sortdir = ''
    aantal = len(os.listdir(path))
    teller = 0
    for i in range(len(img_dir)):
        filename = img_dir[i]
        img = skimage.io.imread(f'{path}{filename}')
        img = skimage.transform.resize(img, (64,64))
        img = img.reshape((1, -1))
        if kmeans.predict(img) == 1:
            sortdir = 'positive'
        else:
            sortdir = 'negative'
        
        shutil.copy(f"{path}/{filename}", f"{sorted_path}/{sortdir}/{filename}")
        
        teller += 1
        # if teller % 100 == 0:
        print(f'{teller}/{aantal}', end='\r')
    # print(f'{teller}/{aantal}', end='\r')

In [4]:
def split_in_four_512(img_dir, label_dir, img_splitted, labels_splitted):
    # Remove the directory so we can stat anew
    if os.path.isdir(img_splitted):
        shutil.rmtree(img_splitted)
    os.makedirs(img_splitted)
    if os.path.isdir(labels_splitted):
        shutil.rmtree(labels_splitted)
    os.makedirs(labels_splitted)

    aantal = len(os.listdir(img_dir))
    teller_files = 0

    for filename in os.listdir(img_dir):
        name_short = filename.split(".")[0]
        
        im = skimage.io.imread(f'{img_dir}/{name_short}.JPG')
        M = im.shape[0]//2
        N = im.shape[1]//2
        tiles = [im[x:x+M,y:y+N] for x in range(0,im.shape[0],M) for y in range(0,im.shape[1],N)]
        try:
            labels = open(f"{label_dir}/{name_short}.txt")
            lines = [line.rstrip() for line in labels.readlines()]
        except:
            lines = []
        for i in range(1, len(tiles)+1):
            skimage.io.imsave(f"{img_splitted}/{name_short}_{i}.JPG", tiles[i-1])
            labels_part = open(f"{labels_splitted}/{name_short}_{i}.txt", "a")
            for line in lines:
                name, x_center, y_center, width, height = [float(s) for s in line.split("\t")]
                if i == 1 and x_center < M and y_center < N:
                    labels_part.write("0\t"+str(x_center)+"\t"+str(y_center)+"\t"+str(width)+"\t"+str(height)+"\n")
                elif i == 2 and x_center > M and y_center < N:
                    labels_part.write("0\t"+str((x_center-M))+"\t"+str(y_center)+"\t"+str(width)+"\t"+str(height)+"\n")
                elif i == 3 and x_center < M and y_center > N:
                    labels_part.write("0\t"+str(x_center)+"\t"+str((y_center-N))+"\t"+str(width)+"\t"+str(height)+"\n")
                elif i == 4 and x_center > M and y_center > N:
                    labels_part.write("0\t"+str((x_center-M))+"\t"+str((y_center-N))+"\t"+str(width)+"\t"+str(height)+"\n")
            labels_part.close()
            if os.stat(labels_part.name).st_size == 0: os.remove(labels_part.name)
        teller_files += 1
        # if(not teller_files%100):
        print(f"{teller_files}/{aantal}", end='\r')
    # print(f"{teller_files}/{aantal}", end='\r')

In [5]:
def split_in_four(img_dir, label_dir, img_splitted, labels_splitted):
    # Remove the directory so we can stat anew
    if os.path.isdir(img_splitted):
        shutil.rmtree(img_splitted)
    os.makedirs(img_splitted)
    if os.path.isdir(labels_splitted):
        shutil.rmtree(labels_splitted)
    os.makedirs(labels_splitted)

    aantal = len(os.listdir(img_dir))
    teller_files = 0

    for filename in os.listdir(img_dir):
        name_short = filename.split(".")[0]
        if f'{name_short}.txt' in os.listdir(label_dir):
        
            im = skimage.io.imread(f'{img_dir}/{name_short}.JPG')
            M = im.shape[0]//2
            N = im.shape[1]//2
            tiles = [im[x:x+M,y:y+N] for x in range(0,im.shape[0],M) for y in range(0,im.shape[1],N)]
            try:
                labels = open(f"{label_dir}/{name_short}.txt")
                lines = [line.rstrip() for line in labels.readlines()]
            except:
                lines = []
            for i in range(1, len(tiles)+1):
                skimage.io.imsave(f"{img_splitted}/{name_short}_{i}.JPG", tiles[i-1])
                labels_part = open(f"{labels_splitted}/{name_short}_{i}.txt", "a")
                for line in lines:
                    name, x_center, y_center, width, height = [float(s) for s in line.split("\t")]
                    if i == 1 and x_center < M and y_center < N:
                        labels_part.write("0\t"+str(x_center)+"\t"+str(y_center)+"\t"+str(width)+"\t"+str(height)+"\n")
                    elif i == 2 and x_center > M and y_center < N:
                        labels_part.write("0\t"+str((x_center-M))+"\t"+str(y_center)+"\t"+str(width)+"\t"+str(height)+"\n")
                    elif i == 3 and x_center < M and y_center > N:
                        labels_part.write("0\t"+str(x_center)+"\t"+str((y_center-N))+"\t"+str(width)+"\t"+str(height)+"\n")
                    elif i == 4 and x_center > M and y_center > N:
                        labels_part.write("0\t"+str((x_center-M))+"\t"+str((y_center-N))+"\t"+str(width)+"\t"+str(height)+"\n")
                labels_part.close()
                if os.stat(labels_part.name).st_size == 0: os.remove(labels_part.name)
        teller_files += 1
        # if(not teller_files%100):
        print(f"{teller_files}/{aantal}", end='\r')
    # print(f"{teller_files}/{aantal}", end='\r')

In [6]:
def prepare_model_data(dir_data, dir_labels, train_path, test_path):
    train_test_split = 0.8

    # Remove the directories so we can stat anew
    if os.path.isdir(train_path):
        shutil.rmtree(train_path)
    os.makedirs(train_path)
    os.makedirs(f"{train_path}/positive")
    os.makedirs(f"{train_path}/negative")
    if os.path.isdir(test_path):
        shutil.rmtree(test_path)
    os.makedirs(test_path)
    os.makedirs(f"{test_path}/positive")
    os.makedirs(f"{test_path}/negative")


    dir_list = os.listdir(dir_data)
    label_list = os.listdir(dir_labels)
    parentdir = train_path
    teller = 0
    aantal = len(dir_list)

    for filename in dir_list:
        labelname = filename.split(".")[0] + ".txt"
        if labelname in label_list:
            shutil.copy(f"{dir_data}/{filename}", f"{parentdir}/positive/{filename}")
            # im=Image.open(f"{dir_data_rescaled}/{filename}")
            # im_blur=im.filter(ImageFilter.GaussianBlur(0.5))
            # im_unsharp=im.filter(ImageFilter.UnsharpMask(0.5))
            # im_blur.save(f"{parentdir}/positive/{filename.split('.')[0]}_b.JPG")
            # im_unsharp.save(f"{parentdir}/positive/{filename.split('.')[0]}_ush.JPG")
        else:
            shutil.copy(f"{dir_data}/{filename}", f"{parentdir}/negative/{filename}")

        rnd = random.random()
        if rnd <= train_test_split:
            parentdir = train_path
        else:
            parentdir = test_path

        teller += 1
        # if(not teller%100):
        print(f"{teller}/{aantal}", end='\r')
    # print(f"{teller}/{aantal}", end='\r')

In [8]:
img_to_512("./data/images_full/", "./data/labels_full_txt/",
           './data/images_512/', './data/label_512/')

1/77
2/77
3/77
4/77
5/77
6/77
7/77
8/77
9/77
10/77
11/77
12/77
13/77
14/77
15/77
16/77
17/77
18/77
19/77
20/77
21/77
22/77
23/77
24/77
25/77
26/77
27/77
28/77
29/77
30/77
31/77
32/77
33/77
34/77
35/77
36/77
37/77
38/77
39/77
40/77
41/77
42/77
43/77
44/77
45/77
46/77
47/77
48/77
49/77
50/77
51/77
52/77
53/77
54/77
55/77
56/77
57/77
58/77
59/77
60/77
61/77
62/77
63/77
64/77
65/77
66/77
67/77
68/77
69/77
70/77
71/77
72/77
73/77
74/77
75/77
76/77
77/77


In [9]:
split_clustering('./data/images_512/', './data/images_512_kmeans/')

c:\Python310\lib\site-packages\sklearn\base.py:288: UserWarning: Trying to unpickle estimator KMeans from version 1.0.2 when using version 1.2.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


100/11550
200/11550
300/11550
400/11550
500/11550
600/11550
700/11550
800/11550
900/11550
1000/11550
1100/11550
1200/11550
1300/11550
1400/11550
1500/11550
1600/11550
1700/11550
1800/11550
1900/11550
2000/11550
2100/11550
2200/11550
2300/11550
2400/11550
2500/11550
2600/11550
2700/11550
2800/11550
2900/11550
3000/11550
3100/11550
3200/11550
3300/11550
3400/11550
3500/11550
3600/11550
3700/11550
3800/11550
3900/11550
4000/11550
4100/11550
4200/11550
4300/11550
4400/11550
4500/11550
4600/11550
4700/11550
4800/11550
4900/11550
5000/11550
5100/11550
5200/11550
5300/11550
5400/11550
5500/11550
5600/11550
5700/11550
5800/11550
5900/11550
6000/11550
6100/11550
6200/11550
6300/11550
6400/11550
6500/11550
6600/11550
6700/11550
6800/11550
6900/11550
7000/11550
7100/11550
7200/11550
7300/11550
7400/11550
7500/11550
7600/11550
7700/11550
7800/11550
7900/11550
8000/11550
8100/11550
8200/11550
8300/11550
8400/11550
8500/11550
8600/11550
8700/11550
8800/11550
8900/11550
9000/11550
9100/11550
9200/115

In [7]:
split_in_four_512('./data/images_512_kmeans/positive/', './data/label_512/',
              './data/images_256/', './data/label_256/')

In [8]:
split_in_four('./data/images_256/', './data/label_256/',
              './data/images_128/', './data/label_128/')

In [9]:
split_in_four('./data/images_128/', './data/label_128/',
              './data/images_64/', './data/label_64/')

In [10]:
prepare_model_data('./data/images_64/', './data/label_64/',
                   './data/images_64_train/', './data/images_64_test/')

In [11]:
prepare_model_data('./data/images_128/', './data/label_128/',
                   './data/images_128_train/', './data/images_128_test/')

In [12]:
prepare_model_data('./data/images_256/', './data/label_256/',
                   './data/images_256_train/', './data/images_256_test/')